In [12]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, make_scorer
import pandas as pd
from sklearn.utils import shuffle
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
import os
import numpy as np

In [20]:
tcga_path = "/home/wollerf/Projects/graph_based_embeddings.git/data/preprocessed/graph_based/TCGA_LUAD/embeddings"
tcga_targets = pd.read_csv("/home/wollerf/Projects/graph_based_embeddings.git/data/input/TCGA_LUAD_targets.csv", index_col=0)
print(tcga_targets)
result_dict = {"Method": [], "Target": [], "Type": [], "Score": []}
cat_targets = ["Disease Free Status", "Disease-specific Survival status", "Progression Free Status"]
num_targets = ["Disease Free (Months)", "Months of disease-specific survival", "Progress Free Survival (Months)"]
target_relabel = {"Disease Free Status" : "Disease Free Status", "Disease-specific Survival status" : "DSS Status", "Progression Free Status" : "Progression Free Status", "Disease Free (Months)" : "Diseases Free Time", "Months of disease-specific survival" : "DSS Time", "Progress Free Survival (Months)" : "Progression Free Survival Time"}

                 Disease Free (Months)  Disease Free Status  \
Sample ID                                                     
TCGA-05-4244-01                    NaN                  NaN   
TCGA-05-4249-01                    NaN                  NaN   
TCGA-05-4250-01                    NaN                  NaN   
TCGA-05-4382-01              10.980702                  1.0   
TCGA-05-4384-01                    NaN                  NaN   
...                                ...                  ...   
TCGA-NJ-A55O-01                    NaN                  NaN   
TCGA-NJ-A55R-01                    NaN                  NaN   
TCGA-NJ-A7XG-01                    NaN                  NaN   
TCGA-O1-A52J-01                    NaN                  NaN   
TCGA-S2-AA1A-01              16.865569                  0.0   

                 Months of disease-specific survival  \
Sample ID                                              
TCGA-05-4244-01                             0.000000   
TCGA-05-4249

In [21]:
NUM_DIMENSION = 64
for i in range(10):
    # Load embeddings of all three datasets.
    tcga_file = os.path.join(tcga_path, f"TCGA_LUAD_samples_{NUM_DIMENSION}_{i}.tsv")
    tcga_umap_file = os.path.join(tcga_path, f"TCGA_LUAD_UMAP_{NUM_DIMENSION}_{i}.csv")
    
    tcga_df = pd.read_csv(tcga_file, sep='\t', index_col=0)
    tcga_umap = pd.read_csv(tcga_umap_file, index_col=0)
    
    tcga_df = tcga_df.join(tcga_targets)
    tcga_umap = tcga_umap.join(tcga_targets)
    
    # Init regression and CV models.
    clf = LogisticRegression(
    penalty="l2",
    solver="liblinear",
    max_iter=1000
    )

    cv = StratifiedKFold(
        n_splits=10,
        shuffle=True,
        random_state=42
    )

    for cat_var in cat_targets:
        # Separability analysis for vital status.
        tcga_vital_mask = tcga_df[cat_var].notna()
        tcga_vital_df = tcga_df.loc[tcga_vital_mask].copy()
        tcga_vital_X = tcga_vital_df.iloc[:, :NUM_DIMENSION].values
        tcga_vital_y = tcga_vital_df[cat_var].values
        tcga_umap_vital_mask = tcga_umap[cat_var].notna()
        tcga_umap_vital_df = tcga_umap.loc[tcga_umap_vital_mask].copy()
        tcga_umap_vital_X = tcga_umap_vital_df.iloc[:, :NUM_DIMENSION].values
        tcga_umap_vital_y = tcga_umap_vital_df[cat_var].values
        # Compute positive class ratio.
        pos_class_ratio = tcga_vital_y.mean()
        
        pome_vital_auc_scores = cross_val_score(
            clf,
            tcga_vital_X,
            tcga_vital_y,
            cv=cv,
            scoring="average_precision"
        )    
        
        umap_vital_auc_scores = cross_val_score(
            clf,
            tcga_umap_vital_X,
            tcga_umap_vital_y,
            cv=cv,
            scoring="average_precision"
        )    
        
        assert len(pome_vital_auc_scores)==len(umap_vital_auc_scores)
        for i in range(len(pome_vital_auc_scores)):
            pome_auc = pome_vital_auc_scores[i]
            umap_auc = umap_vital_auc_scores[i]
            result_dict["Method"].append("POME")
            result_dict["Target"].append(cat_var)
            result_dict["Score"].append(pome_auc)
            result_dict["Type"].append("AP")
            result_dict["Method"].append("UMAP")
            result_dict["Target"].append(cat_var)
            result_dict["Score"].append(umap_auc)
            result_dict["Type"].append("AP")
        
        result_dict["Method"].append("Naive")
        result_dict["Target"].append(cat_var)
        result_dict["Score"].append(pos_class_ratio)
        result_dict["Type"].append("AP")
        
        if False: # uncomment if desired to compute random background model
            perm_vital_auc = []

            for _ in range(100):
                y_perm = shuffle(tcga_vital_y, random_state=None)
                score = cross_val_score(
                    clf,
                    tcga_vital_X,
                    y_perm,
                    cv=cv,
                    scoring="average_precision"
                ).mean()
                perm_vital_auc.append(score)

            for i in range(len(perm_vital_auc)):
                auc = perm_vital_auc[i]
                result_dict["Method"].append("Random(POME)")
                result_dict["Target"].append(cat_var)
                result_dict["Type"].append("AUC")
                result_dict["Score"].append(auc)
                
            perm_vital_auc_umap = []

            for _ in range(100):
                y_perm = shuffle(tcga_umap_vital_y, random_state=None)
                score = cross_val_score(
                    clf,
                    tcga_umap_vital_X,
                    y_perm,
                    cv=cv,
                    scoring="average_precision"
                ).mean()
                perm_vital_auc_umap.append(score)

            for i in range(len(perm_vital_auc_umap)):
                auc = perm_vital_auc_umap[i]
                result_dict["Method"].append("Random(UMAP)")
                result_dict["Target"].append(cat_var)
                result_dict["Type"].append("AUC")
                result_dict["Score"].append(auc)
        
    #### Separability analysis for numeric targets ####
    reg = LinearRegression()

    cv = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
    )
    
    for num_var in num_targets:
        tcga_days_mask = tcga_df[num_var].notna()
        tcga_days_df = tcga_df.loc[tcga_days_mask].copy()

        tcga_days_X = tcga_days_df.iloc[:, :NUM_DIMENSION].values
        tcga_days_y = tcga_days_df[num_var].values

        tcga_umap_days_mask = tcga_umap[num_var].notna()
        tcga_umap_days_df = tcga_umap.loc[tcga_umap_days_mask]

        tcga_days_umap_X = tcga_umap_days_df.iloc[:, :NUM_DIMENSION].values
        tcga_days_umap_y = tcga_umap_days_df[num_var].values
        
        pome_days_r2_scores = cross_val_score(
        reg,
        tcga_days_X,
        tcga_days_y,
        cv=cv,
        scoring="r2"
        )

        umap_days_r2_scores = cross_val_score(
        reg,
        tcga_days_umap_X,
        tcga_days_umap_y,
        cv=cv,
        scoring="r2"
        )
        
        assert len(pome_days_r2_scores) == len(umap_days_r2_scores)
        for i in range(len(pome_days_r2_scores)):
            result_dict["Method"].append("POME")
            result_dict["Target"].append(num_var)
            result_dict["Score"].append(pome_days_r2_scores[i])
            result_dict["Type"].append("R2")

            result_dict["Method"].append("UMAP")
            result_dict["Target"].append(num_var)
            result_dict["Score"].append(umap_days_r2_scores[i])
            result_dict["Type"].append("R2")

        if False:
            perm_days_r2_scores = []

            for _ in range(100):
                y_perm = shuffle(tcga_days_y, random_state=None)

                score = cross_val_score(
                    reg,
                    tcga_days_X,
                    y_perm,
                    cv=cv,
                    scoring="r2"
                ).mean()

                perm_days_r2_scores.append(score)
            
            for score in perm_days_r2_scores:
                result_dict["Method"].append("Random(POME)")
                result_dict["Target"].append(num_var)
                result_dict["Score"].append(score)
                result_dict["Type"].append("R2")
                
            perm_umap_scores = []

            for _ in range(100):
                y_perm = shuffle(tcga_days_umap_y, random_state=None)

                score = cross_val_score(
                    reg,
                    tcga_days_umap_X,
                    y_perm,
                    cv=cv,
                    scoring="r2"
                ).mean()

                perm_umap_scores.append(score)
            
            for score in perm_umap_scores:
                result_dict["Method"].append("Random(UMAP)")
                result_dict["Target"].append(num_var)
                result_dict["Score"].append(score)
                result_dict["Type"].append("R2")

In [22]:
result_df = pd.DataFrame(result_dict)
result_df
result_df.to_csv(f"TCGA_LUAD_regression_results_{NUM_DIMENSION}.csv", index=False)